匯入套件

In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import geopandas as gpd

合併111到113年交通事故紀錄

In [47]:
import glob
import os

# 中文字體設定（讓圖表顯示中文）
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False

# --- ✅ 明確列出三個檔案 ---
files = [
    './accident_111_taipei.csv',
    './accident_112_taipei.csv',
    './accident_113_taipei.csv'
]

print("偵測到的檔案：", files)

# --- 讀取並合併所有年度資料 ---
dfs = []
for file in files:
    year = os.path.basename(file).split('_')[1]
    print(f"🔹 讀取 {year} 年資料中...")
    df = pd.read_csv(file, encoding='big5')
    df['年份'] = int(year)
    dfs.append(df)

# --- 合併為一份 DataFrame ---
df_all = pd.concat(dfs, ignore_index=True)
print(f"\n已成功合併 {len(files)} 份資料，共 {df_all.shape[0]} 筆紀錄\n")

# --- 分組與彙整 ---
grouped_df = (
    df_all.groupby(['年份', '發生月', '發生日', '發生時-Hours', '發生分', '肇事地點'])
          .agg({
              '死亡人數': 'sum',
              '受傷人數': 'sum',
              '事故類型及型態': 'first',
              '區序': 'first',
              '天候': 'first',
              '道路型態': 'first',
              '座標-X': 'first',
              '座標-Y': 'first',
          })
          .reset_index()
)

grouped_df['傷亡人數'] = grouped_df['死亡人數'] + grouped_df['受傷人數']

print("合併後欄位：", list(grouped_df.columns))
print("筆數：", len(grouped_df))

# --- 輸出結果 ---
output_path = './accident_111to113_grouped.xlsx'
grouped_df.to_excel(output_path, index=False)
print(f"\n已成功輸出整合檔案：{output_path}")


偵測到的檔案： ['./accident_111_taipei.csv', './accident_112_taipei.csv', './accident_113_taipei.csv']
🔹 讀取 111 年資料中...


C:\Users\dcfoa\AppData\Local\Temp\ipykernel_10180\2972389816.py:22: DtypeWarning: Columns (43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='big5')


🔹 讀取 112 年資料中...


C:\Users\dcfoa\AppData\Local\Temp\ipykernel_10180\2972389816.py:22: DtypeWarning: Columns (43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, encoding='big5')


🔹 讀取 113 年資料中...

已成功合併 3 份資料，共 166162 筆紀錄

合併後欄位： ['年份', '發生月', '發生日', '發生時-Hours', '發生分', '肇事地點', '死亡人數', '受傷人數', '事故類型及型態', '區序', '天候', '道路型態', '座標-X', '座標-Y', '傷亡人數']
筆數： 72105

已成功輸出整合檔案：./accident_111to113_grouped.xlsx


預測模型：未來24小時車禍風險
主要目的：以時間與空間分布預測未來 24 小時高風險熱點

In [48]:
# ===================================================
# accident_forecast_model.py (進階版)
# 功能：
# - 根據三年資料計算風險預測分數
# - 保留「肇事地點」、「年份」、「事故嚴重程度(A1/A2)」
# ===================================================

import pandas as pd
from datetime import datetime, timedelta

# ---------- 1️⃣ 讀取資料 ----------
input_path = "./accident_111to113_grouped.xlsx"
df = pd.read_excel(input_path)
print("✅ 成功讀取資料，共", df.shape[0], "筆")

# ---------- 2️⃣ 欄位統一 ----------
df = df.rename(columns={
    "區序": "district",
    "發生時-Hours": "hour",
    "座標-X": "x",
    "座標-Y": "y",
    "肇事地點": "location"
})

# 移除缺失與無效資料
df = df.dropna(subset=["district", "x", "y", "hour"])
df["hour"] = df["hour"].astype(int)

# ---------- 3️⃣ 新增欄位：年份與事故嚴重程度 ----------
# 由「年份」欄直接取值
if "年份" in df.columns:
    df["year"] = df["年份"]
else:
    df["year"] = datetime.now().year - 1911  # 若缺失，預設為最新民國年

# 判斷嚴重程度：若欄位中包含「死亡」字樣 → A1，否則 A2
if "事故類型及型態" in df.columns:
    df["severity"] = df["事故類型及型態"].apply(lambda x: "A1" if "死亡" in str(x) else "A2")
else:
    df["severity"] = "A2"

# ---------- 4️⃣ 時間風險 ----------
district_totals = df.groupby("district").size().rename("total")
hour_risk = (
    df.groupby(["district", "hour"])
      .size()
      .reset_index(name="count")
      .merge(district_totals, on="district", how="left")
)
hour_risk["hour_risk"] = hour_risk["count"] / hour_risk["total"]

# ---------- 5️⃣ 空間風險 ----------
bin_size = 0.001
df["x_bin"] = (df["x"] / bin_size).round()
df["y_bin"] = (df["y"] / bin_size).round()

# 同格網取最常見地點 + 保留年與嚴重度的統計特徵
space_risk = (
    df.groupby(["district", "x_bin", "y_bin", "year", "severity"])
      .agg({
          "location": lambda x: x.value_counts().index[0],
          "x": "mean",
          "y": "mean",
          "hour": "count"
      })
      .rename(columns={"hour": "count"})
      .reset_index()
)

# 區域內比例正規化
dist_total = space_risk.groupby("district")["count"].sum().rename("total")
space_risk = space_risk.merge(dist_total, on="district", how="left")
space_risk["space_risk"] = space_risk["count"] / space_risk["total"]

# ---------- 6️⃣ 預測未來 24 小時 ----------
future_hours = [(datetime.now() + timedelta(hours=i)).hour for i in range(1, 25)]

forecast = (
    hour_risk[hour_risk["hour"].isin(future_hours)]
    .merge(space_risk, on="district", how="inner")
)

# ---------- 7️⃣ 計算綜合風險分數 ----------
forecast["risk_score"] = forecast["hour_risk"] * forecast["space_risk"]
max_rs = forecast["risk_score"].max()
forecast["risk_score"] = forecast["risk_score"] / max_rs if max_rs > 0 else 0.0

# 加上實際座標
forecast["lat"] = forecast["y"]
forecast["lng"] = forecast["x"]

# ---------- 8️⃣ 輸出結果 ----------
output_path = "./forecast_result.csv"
forecast.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"💾 已輸出預測結果：{output_path}")
print("📋 欄位包含：", list(forecast.columns))

✅ 成功讀取資料，共 72105 筆
💾 已輸出預測結果：./forecast_result.csv
📋 欄位包含： ['district', 'hour', 'count_x', 'total_x', 'hour_risk', 'x_bin', 'y_bin', 'year', 'severity', 'location', 'x', 'y', 'count_y', 'total_y', 'space_risk', 'risk_score', 'lat', 'lng']


預測結果地圖輸出 (HTML)
主要目的：將未來24小時高風險熱點在地圖上可視化

In [49]:
# ===================================================
# accident_forecast_map_v11.py
# 功能：
# ✅ 每行政區前5筆最高風險地點
# ✅ 保留肇事地點名稱
# ✅ 清除所有按鈕 + 全螢幕 + 色條
# ✅ 去掉動態時間滑條（TimestampedGeoJson）
# ===================================================

import pandas as pd
import folium
from folium import FeatureGroup, LayerControl
from folium.plugins import Fullscreen
import branca.colormap as cm
import numpy as np

# ---------- 1️⃣ 讀取資料 ----------
forecast = pd.read_csv("./forecast_result.csv")

# ---------- 2️⃣ 欄位檢查 ----------
required_cols = ["district", "hour", "lat", "lng", "risk_score"]
for c in required_cols:
    if c not in forecast.columns:
        raise ValueError(f"⚠️ 缺少必要欄位：{c}")

if "location" in forecast.columns:
    location_col = "location"
elif "肇事地點" in forecast.columns:
    location_col = "肇事地點"
else:
    location_col = None
    print("⚠️ 找不到地點欄位，將不顯示地點名稱。")

print(f"✅ 成功載入 {forecast.shape[0]} 筆資料")

# ---------- 3️⃣ 每行政區取前5筆 ----------
forecast_top = (
    forecast.sort_values(["district", "risk_score"], ascending=[True, False])
            .groupby("district", as_index=False)
            .head(5)
            .copy()
)

# ---------- 4️⃣ 防重疊 ----------
offset_range = 0.00025
forecast_top["lat_jitter"] = forecast_top["lat"] + np.random.uniform(-offset_range, offset_range, len(forecast_top))
forecast_top["lng_jitter"] = forecast_top["lng"] + np.random.uniform(-offset_range, offset_range, len(forecast_top))

# ---------- 5️⃣ 地圖設定 ----------
center = [25.04, 121.54]
m = folium.Map(location=center, zoom_start=12, tiles="CartoDB Positron")
Fullscreen(position="topright").add_to(m)

# 色條
cmap = cm.linear.YlOrRd_09.scale(0, 1)
cmap.caption = "風險值（相對分數 0–1）"
cmap.add_to(m)

popup_width = 260
radius_base = 5
radius_scale = 12

# ---------- 6️⃣ 行政區層 ----------
districts = sorted(forecast_top["district"].unique())

for dist in districts:
    dist_fg = FeatureGroup(name=f"🏙️ {dist}（前5熱點）", show=False)
    data_dist = forecast_top[forecast_top["district"] == dist]

    for _, row in data_dist.iterrows():
        r = float(row["risk_score"])
        place_name = row[location_col] if location_col else "（無地點資訊）"
        popup_html = f"""
        <div style='width:{popup_width}px; font-size:13px; line-height:1.5;'>
            <b>行政區：</b>{row['district']}<br>
            <b>肇事地點：</b>{place_name}<br>
            <b>風險值：</b>{r:.3f}<br>
            <b>時段：</b>{int(row['hour'])} 時
        </div>
        """
        folium.CircleMarker(
            location=[row["lat_jitter"], row["lng_jitter"]],
            radius=radius_base + r * radius_scale,
            color=cmap(r),
            fill=True,
            fill_color=cmap(r),
            fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=popup_width + 40),
        ).add_to(dist_fg)

    dist_fg.add_to(m)

# ---------- 7️⃣ 清除按鈕 ----------
clear_js = """
function clearAllLayers() {
    var inputs = document.querySelectorAll('.leaflet-control-layers-selector');
    inputs.forEach(function(input) {
        if (input.checked) { input.click(); }
    });
}
"""
m.get_root().html.add_child(folium.Element(f"<script>{clear_js}</script>"))
m.get_root().html.add_child(folium.Element(
    "<button onclick='clearAllLayers()' "
    "style='position: fixed; top: 10px; left: 10px; z-index: 9999; background: #fff; "
    "border: 1px solid #ccc; padding: 6px 12px; border-radius: 5px;'>🧹 清除所有</button>"
))

# ---------- 8️⃣ 輸出 ----------
LayerControl(collapsed=False, position="topright").add_to(m)
output_path = "./Taipei_accident_forecast_v9.html"
m.save(output_path)
print(f"🌐 已輸出地圖：{output_path}")
m


✅ 成功載入 453792 筆資料
🌐 已輸出地圖：./Taipei_accident_forecast_v9.html
